In [1]:
!pip install numpy==1.26.4 --quiet
!pip install torch torchvision torchaudio --quiet
!pip install torch-geometric --quiet
!pip install networkx sentence-transformers --quiet
!pip install torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.3 MB/s eta 0:00:00a 0:00:01


In [2]:
import torch

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [31]:
import glob
import logging
import os
import shutil

import torch
from torch_geometric.data import Data
from torch_geometric.datasets import SNAPDataset
from torch_geometric.utils import to_undirected

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# SNAP ego-Twitter raw: *.circles, *.edges, *.egofeat, *.feat, *.featnames
# Kaggle: /kaggle/input/datasets/akshatsharma1011/twitter/twitter
SNAP_ROOT = os.environ.get("SNAP_DATA_ROOT", "/kaggle/working/snap_twitter")
TWITTER_SNAP_SOURCE = os.environ.get(
    "TWITTER_SNAP_SOURCE",
    "/kaggle/input/datasets/akshatsharma1011/twitter/twitter",
)
FEATURE_DIM = 128


def _ego_twitter_raw_dir() -> str:
    return os.path.join(SNAP_ROOT, "ego-twitter", "raw")


def prepare_snap_twitter_raw() -> bool:
    """Copy Kaggle SNAP files into PyG raw/; remove processed cache if we copied."""
    if not os.path.isdir(TWITTER_SNAP_SOURCE):
        logger.info(
            "SNAP source not found: %s (CSV fallback, PyG download, or set TWITTER_SNAP_SOURCE).",
            TWITTER_SNAP_SOURCE,
        )
        return False
    raw_dir = _ego_twitter_raw_dir()
    os.makedirs(raw_dir, exist_ok=True)
    n = 0
    for path in glob.glob(os.path.join(TWITTER_SNAP_SOURCE, "*")):
        if os.path.isfile(path):
            shutil.copy2(path, os.path.join(raw_dir, os.path.basename(path)))
            n += 1
    processed_pt = os.path.join(SNAP_ROOT, "ego-twitter", "processed", "data.pt")
    if os.path.isfile(processed_pt):
        os.remove(processed_pt)
    logger.info("Copied %d files from %s -> %s", n, TWITTER_SNAP_SOURCE, raw_dir)
    return n > 0


def _raw_dir_has_files() -> bool:
    d = _ego_twitter_raw_dir()
    if not os.path.isdir(d):
        return False
    return any(os.path.isfile(os.path.join(d, f)) for f in os.listdir(d))


def assign_community_labels(edge_index: torch.Tensor, num_nodes: int) -> torch.Tensor:
    degrees = torch.zeros(num_nodes)
    degrees.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))
    degrees.scatter_add_(0, edge_index[1], torch.ones(edge_index.size(1)))
    q33 = degrees.quantile(0.33).item()
    q66 = degrees.quantile(0.66).item()
    q90 = degrees.quantile(0.90).item()
    labels = torch.zeros(num_nodes, dtype=torch.long)
    labels[degrees > q33] = 1
    labels[degrees > q66] = 2
    labels[degrees > q90] = 3
    return labels


def build_structural_features(edge_index: torch.Tensor, num_nodes: int, feature_dim: int) -> torch.Tensor:
    deg = torch.zeros(num_nodes)
    deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))
    deg.scatter_add_(0, edge_index[1], torch.ones(edge_index.size(1)))
    deg_norm = (deg - deg.mean()) / (deg.std() + 1e-8)
    remaining = feature_dim - 1
    rnd = torch.randn(num_nodes, remaining) * 0.1
    return torch.cat([deg_norm.unsqueeze(1), rnd], dim=1)


def load_twitter_ego() -> Data:
    copied = prepare_snap_twitter_raw()
    kaggle_path = "/kaggle/input/twitter-social-circles/twitter_edges.csv"
    if (not _raw_dir_has_files()) and os.path.exists(kaggle_path):
        import pandas as pd
        df = pd.read_csv(kaggle_path)
        src = torch.tensor(df.iloc[:, 0].values, dtype=torch.long)
        dst = torch.tensor(df.iloc[:, 1].values, dtype=torch.long)
        edge_index = to_undirected(torch.stack([src, dst]))
        num_nodes = int(edge_index.max().item()) + 1
        x = build_structural_features(edge_index, num_nodes, FEATURE_DIM)
        y = assign_community_labels(edge_index, num_nodes)
        return Data(x=x, edge_index=edge_index, y=y, num_nodes=num_nodes)
    if _raw_dir_has_files() or copied:
        logger.info(
            "Loading SNAPDataset from %s (force_reload=%s).",
            _ego_twitter_raw_dir(),
            copied,
        )
        dataset = SNAPDataset(root=SNAP_ROOT, name="ego-Twitter", force_reload=copied)
    else:
        logger.info("Loading PyG SNAPDataset ego-Twitter (first run downloads)...")
        dataset = SNAPDataset(root=SNAP_ROOT, name="ego-Twitter")
    all_edges_src, all_edges_dst = [], []
    offset = 0
    for d in dataset:
        if d.edge_index.size(1) == 0:
            continue
        all_edges_src.append(d.edge_index[0] + offset)
        all_edges_dst.append(d.edge_index[1] + offset)
        offset += d.num_nodes
    edge_src = torch.cat(all_edges_src)
    edge_dst = torch.cat(all_edges_dst)
    edge_index = to_undirected(torch.stack([edge_src, edge_dst], dim=0))
    num_nodes = offset
    x = build_structural_features(edge_index, num_nodes, FEATURE_DIM)
    y = assign_community_labels(edge_index, num_nodes)
    return Data(x=x, edge_index=edge_index, y=y, num_nodes=num_nodes)


data_pyg = load_twitter_ego()
X = data_pyg.x.numpy()
Y = data_pyg.y.numpy().astype(int)
edge_index = data_pyg.edge_index.contiguous()
print("Nodes:", data_pyg.num_nodes, "| X:", X.shape, "| edges:", edge_index.shape[1])


INFO:__main__:Copied 4865 files from /kaggle/input/datasets/akshatsharma1011/twitter/twitter -> /kaggle/working/snap_twitter/ego-twitter/raw
INFO:__main__:Loading SNAPDataset from /kaggle/working/snap_twitter/ego-twitter/raw (force_reload=True).
Processing...
  0%|          | 0/973 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch_geometric/datasets/snap_dataset.py:72: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  x = x.to_sparse_csr()
100%|██████████| 973/973 [02:55<00:00,  5.54it/s]
Done!
/usr/local/lib/python3.12/dist-packages/torch_geometric/data/in_memory_dataset.py:131: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([torc

Nodes: 133857 | X: (133857, 128) | edges: 3549272


In [32]:
from collections import Counter
print(Counter(Y).most_common(10))


[(0, 46354), (1, 43258), (2, 31229), (3, 13016)]


In [33]:
print("Edge index shape:", edge_index.shape)


Edge index shape: torch.Size([2, 3549272])


In [34]:
from torch_geometric.data import Data

x = torch.tensor(X, dtype=torch.float)
y = torch.tensor(Y, dtype=torch.long)
data = Data(x=x, edge_index=edge_index, y=y).to(device)
print(data)


Data(x=[133857, 128], edge_index=[2, 3549272], y=[133857])


In [35]:
from collections import Counter
from sklearn.model_selection import train_test_split
import numpy as np
import torch

counts = Counter(Y)
valid_classes = {cls for cls, cnt in counts.items() if cnt >= 2}
valid_indices = [i for i, label in enumerate(Y) if label in valid_classes]
Y_filtered = np.array([Y[i] for i in valid_indices])
if len(set(Y_filtered)) < 2:
    train_idx, test_idx = train_test_split(valid_indices, test_size=0.2, random_state=42)
else:
    train_idx, test_idx = train_test_split(
        valid_indices, test_size=0.2, stratify=Y_filtered, random_state=42
    )
train_mask = torch.zeros(len(Y), dtype=torch.bool)
test_mask = torch.zeros(len(Y), dtype=torch.bool)
train_mask[train_idx] = True
test_mask[test_idx] = True
data.train_mask = train_mask
data.test_mask = test_mask
print("Train:", train_mask.sum().item(), "Test:", test_mask.sum().item())


Train: 107085 Test: 26772


In [36]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

model = GraphSAGE(data.num_features, 64, len(set(Y.tolist()))).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)


In [39]:
import torch.nn.functional as F

batch_size = 1024
train_indices = data.train_mask.nonzero(as_tuple=True)[0]

def train():
    model.train()
    total_loss = 0.0
    perm = train_indices[torch.randperm(len(train_indices))]
    steps = 0
    for i in range(0, len(perm), batch_size):
        batch_nodes = perm[i : i + batch_size]
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[batch_nodes], data.y[batch_nodes])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        steps += 1
    return total_loss / max(steps, 1)

for epoch in range(10):
    print(f"Epoch {epoch+1}, Loss: {train():.4f}")


Epoch 1, Loss: 0.0133
Epoch 2, Loss: 0.0126
Epoch 3, Loss: 0.0120
Epoch 4, Loss: 0.0109
Epoch 5, Loss: 0.0115
Epoch 6, Loss: 0.0097
Epoch 7, Loss: 0.0098
Epoch 8, Loss: 0.0085
Epoch 9, Loss: 0.0081
Epoch 10, Loss: 0.0075


In [40]:
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score

def f1_auc_test(y_true, logits):
    y_true = y_true.cpu().numpy()
    proba = torch.softmax(logits, dim=1).cpu().numpy()
    pred = logits.argmax(dim=1).cpu().numpy()
    f1 = f1_score(y_true, pred, average="macro", zero_division=0)
    c = proba.shape[1]
    if c < 2:
        return f1, float("nan")
    try:
        if c == 2:
            auc = roc_auc_score(y_true, proba[:, 1])
        else:
            auc = roc_auc_score(
                y_true, proba, multi_class="ovr", average="macro", labels=np.arange(c)
            )
    except ValueError:
        auc = float("nan")
    return f1, auc


@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
    acc = int(correct) / int(data.test_mask.sum())
    return acc, pred, out

acc, pred, gnn_output = test()
f1, auc = f1_auc_test(data.y[data.test_mask], gnn_output[data.test_mask])
print(f"Test Accuracy: {acc:.4f} | F1 (macro): {f1:.4f} | AUC (macro OVR): {auc:.4f}")


Test Accuracy: 0.9964 | F1 (macro): 0.9943 | AUC (macro OVR): 1.0000


In [41]:
print("Unique predictions:", pred.unique())
print("Unique labels:", data.y.unique())

Unique predictions: tensor([0, 1, 2, 3], device='cuda:0')
Unique labels: tensor([0, 1, 2, 3], device='cuda:0')
